# Bronchoscopy Multi-Agent System Demo Notebook

The system is separated into two lines:

1. **Research pipeline**
   - used for single-turn multi-agent reasoning and multi-turn structured generation
   - better for explainability, structured outputs, and research demos

2. **Runtime pipeline**
   - used for faster, shorter, more stable real-time guidance
   - better suited for online interaction, UI display, and TTS

### 为什么分成两条线路？
最初系统主要依赖 MAS 生成指导内容。MAS 更适合做丰富的多代理推理和结构化输出，但在实时支气管镜引导里，它往往太慢，而且稳定性不够。  
所以现在系统分成两条线：

1. **Research pipeline**
   - 用于单轮多代理推理和多轮结构化输出
   - 更适合研究展示、可解释性和后续扩展

2. **Runtime pipeline**
   - 用于快速、稳定、简短的实时指导生成
   - 更适合在线交互、UI 展示和 TTS

## How to run / 运行说明

This notebook uses the **source code directly**.

1. Unzip the project folder.
2. Open this notebook inside the project root directory.
3. Run the setup cells first.
4. Execute the remaining cells in order.
5. Python version: 3.11

In [ ]:
# Optional: install once if needed
# !pip install smolagents huggingface_hub transformers nbformat

In [1]:
# --- Setup project path ---
import sys
from pathlib import Path

project_root = Path.cwd()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project_root =", project_root)
print("src_path     =", src_path)

project_root = c:\Users\LAB-ADMIN\Desktop\broncho_mas_project_20260317
src_path     = c:\Users\LAB-ADMIN\Desktop\broncho_mas_project_20260317\src


In [2]:
# --- Environment setup ---
import os
import getpass

os.environ.setdefault("BRONCHO_PROVIDER", "hf")
os.environ.setdefault("BRONCHO_MODEL", "Qwen/Qwen3-30B-A3B-Instruct-2507")
os.environ.setdefault("BRONCHO_TOOL_CHOICE", "auto")

if not os.environ.get("HF_TOKEN"):
    try:
        os.environ["HF_TOKEN"] = getpass.getpass("Enter HF_TOKEN: ")
    except Exception:
        print("HF_TOKEN not set in this notebook session.")

print("BRONCHO_PROVIDER =", os.environ.get("BRONCHO_PROVIDER"))
print("BRONCHO_MODEL    =", os.environ.get("BRONCHO_MODEL"))
print("HF_TOKEN exists  =", bool(os.environ.get("HF_TOKEN")))

BRONCHO_PROVIDER = hf
BRONCHO_MODEL    = Qwen/Qwen3-30B-A3B-Instruct-2507
HF_TOKEN exists  = True


In [3]:
print("Testing core imports...")

import broncho_mas
from broncho_mas import SmolAgentsLLM
from broncho_mas.runtime.runtime_manager import RuntimeManager

research_available = True
research_import_error = None

try:
    from broncho_mas.research.manager import MultiAgentManager
except Exception as e:
    research_available = False
    research_import_error = repr(e)

try:
    from broncho_mas.shared.timeline import load_session_metrics
except Exception as e:
    print("Shared timeline import failed:", repr(e))
    load_session_metrics = None

print("broncho_mas imported from:", broncho_mas.__file__)
print("Available top-level names:", [x for x in dir(broncho_mas) if not x.startswith("_")][:30])

if research_available:
    print("Research imports OK.")
else:
    print("Research pipeline not available in this environment.")
    print("Reason:", research_import_error)

Testing core imports...
broncho_mas imported from: c:\Users\LAB-ADMIN\Desktop\broncho_mas_project_20260317\src\broncho_mas\__init__.py
Available top-level names: ['MultiAgentManager', 'RuntimeManager', 'SmolAgentsLLM', 'adapter', 'annotations', 'research', 'runtime', 'shared']
Research imports OK.


## 1) Instantiate managers

We keep both lines visible in the same notebook.

- **Runtime line**: fast structured path for short real-time guidance
- **Research line**: multi-agent path with shared backbone

The research line should keep the **multi-agent system structure** even if not every agent fully shapes the real-time utterance yet.

In [4]:
runtime = RuntimeManager()
print("RuntimeManager created successfully.")
print(runtime)

manager = None
if research_available:
    try:
        manager = MultiAgentManager()
        print("MultiAgentManager created successfully.")
        if hasattr(manager, "logger"):
            print("Research log dir:", getattr(manager.logger, "run_dir", "<no run_dir>"))
    except Exception as e:
        print("Failed to instantiate MultiAgentManager:")
        print(type(e).__name__, str(e))
        manager = None
else:
    print("Research pipeline unavailable in this environment.")

[runtime] provider=hf model=Qwen/Qwen3-30B-A3B-Instruct-2507
RuntimeManager created successfully.
[broncho_mas] log dir = C:\Users\LAB-ADMIN\Desktop\broncho_mas_project_20260317\mas_log\20260317_134824_155998
[broncho_mas manager] version=0.0.4-research research_mode=agentic provider=hf model=Qwen/Qwen3-30B-A3B-Instruct-2507
MultiAgentManager created successfully.
Research log dir: C:\Users\LAB-ADMIN\Desktop\broncho_mas_project_20260317\mas_log\20260317_134824_155998


## 2) Runtime-first synthetic scenarios

These examples use **structured payloads** to test the runtime manager in its main path.

Main checks:
- the manager reads upstream payload directly
- `source_mode` becomes `"payload"`
- runtime outputs stay short and sensible
- the payload already includes fields that are also useful for the tightened shared backbone, such as:
  - `m_jointsVelRel`
  - `distance_delta`
  - `heading_error`

This is still the preferred runtime test path.

In [5]:
from pprint import pprint

def run_runtime_case(case_name, payload, debug=True):
    print("=" * 88)
    print("CASE:", case_name)

    if debug:
        print("INPUT PAYLOAD:")
        pprint(payload)

    result = runtime.step(payload)

    print("\nDISPLAY OUTPUT:")
    print(result.get("ui_text", "<no ui_text>"))

    if debug:
        raw = result.get("raw", {}) or {}

        print("\nSOURCE MODE:")
        print(raw.get("source_mode"))

        print("\nNORMALIZED STATE:")
        pprint(raw.get("normalized_state", {}))

        print("\nPLAN JSON:")
        pprint(result.get("plan_json", {}))

        print("\nSTATISTICS:")
        pprint(result.get("statistics", {}))

        print("\nRAW OUTPUT (selected):")
        pprint({
            "used_fallback_backend": raw.get("used_fallback_backend"),
            "deterministic_ui": raw.get("deterministic_ui"),
            "llm_ui": raw.get("llm_ui"),
        })

    return result

In [6]:
case_1 = {
    "reached_regions": [],
    "anatomical_position": "",
    "current_target": "RB1",
    "is_target_visible": False,
    "is_centered": False,
    "is_stable": False,
    "drift_detected": False,
    "backtracking": False,
    "phase": "navigation",
    "need_llm": True,
    "llm_reason": "Target not yet visible.",
    "soft_prompt": "No target region has been reached yet. Guide the learner toward the next airway.",
    "previous_msgs": "",
    "student_question": "",
    "m_jointsVelRel": [0.0, 0.0, 0.0, 0.04, 0.0],
    "distance_delta": -0.05,
    "heading_error": 0.08,
    "bounding_boxes": [],
}

result_1 = run_runtime_case("Initial orientation", case_1)

CASE: Initial orientation
INPUT PAYLOAD:
{'anatomical_position': '',
 'backtracking': False,
 'bounding_boxes': [],
 'current_target': 'RB1',
 'distance_delta': -0.05,
 'drift_detected': False,
 'heading_error': 0.08,
 'is_centered': False,
 'is_stable': False,
 'is_target_visible': False,
 'llm_reason': 'Target not yet visible.',
 'm_jointsVelRel': [0.0, 0.0, 0.0, 0.04, 0.0],
 'need_llm': True,
 'phase': 'navigation',
 'previous_msgs': '',
 'reached_regions': [],
 'soft_prompt': 'No target region has been reached yet. Guide the learner '
                'toward the next airway.',
 'student_question': ''}

DISPLAY OUTPUT:
Hold center. Look for Mercedes sign or Mercedes sign at the right upper lobe.

SOURCE MODE:
payload

NORMALIZED STATE:
{'backtracking': False,
 'bounding_boxes': [],
 'control_triplet': [0.0, 0.0, 0.0],
 'current_airway': '',
 'drift_detected': False,
 'is_centered': False,
 'is_stable': False,
 'is_target_visible': False,
 'just_reached': False,
 'llm_reason': 'Targe

In [ ]:
case_2 = {
    "reached_regions": ["RB1"],
    "anatomical_position": "RB1",
    "current_target": "RB2",
    "is_target_visible": False,
    "is_centered": False,
    "is_stable": False,
    "drift_detected": True,
    "backtracking": False,
    "phase": "navigation",
    "need_llm": True,
    "llm_reason": "The learner has drifted off target.",
    "soft_prompt": "The learner has drifted off target and needs concise reorientation guidance.",
    "previous_msgs": "",
    "student_question": "",
    "m_jointsVelRel": [0.0, 0.20, 0.0, 0.10, 0.0],
    "distance_delta": -0.28,
    "heading_error": 0.19,
    "bounding_boxes": [],
}

result_2 = run_runtime_case("Drift / reorientation", case_2)

In [ ]:
case_3 = {
    "reached_regions": ["RB1", "RB2"],
    "anatomical_position": "RB2",
    "current_target": "RB3",
    "is_target_visible": True,
    "is_centered": False,
    "is_stable": False,
    "drift_detected": False,
    "backtracking": False,
    "phase": "navigation",
    "need_llm": True,
    "llm_reason": "Target visible but not stable.",
    "soft_prompt": "The target is visible but not yet centered or stable. Give short coaching.",
    "previous_msgs": "",
    "student_question": "",
    "m_jointsVelRel": [0.12, 0.0, 0.0, 0.04, 0.0],
    "distance_delta": -0.10,
    "heading_error": 0.12,
    "bounding_boxes": [],
}

result_3 = run_runtime_case("Visible but unstable target", case_3)

In [ ]:
case_4 = {
    "reached_regions": ["RB1", "RB2", "RB3"],
    "anatomical_position": "RB3",
    "current_target": "RB3",
    "is_target_visible": True,
    "is_centered": True,
    "is_stable": True,
    "drift_detected": False,
    "backtracking": False,
    "phase": "inspection",
    "need_llm": True,
    "llm_reason": "Stable at target.",
    "soft_prompt": "The learner has stabilized at the target. Confirm and transition to the next step.",
    "previous_msgs": "",
    "student_question": "",
    "m_jointsVelRel": [0.0, 0.0, 0.0, 0.0, 0.0],
    "distance_delta": 0.01,
    "heading_error": 0.02,
    "bounding_boxes": [],
}

result_4 = run_runtime_case("Stable at target / transition", case_4)

### Runtime logging smoke test

This checks whether the runtime pipeline actually writes logs to `mas_log/`, whether a timeline file exists, and whether the shared timeline loader can read the run back successfully.

In [ ]:

# Runtime logging smoke test
runtime_logger = getattr(runtime, "logger", None)

if runtime_logger is None:
    print("Runtime manager has no exposed logger attribute.")
else:
    runtime_run_dir = Path(runtime_logger.run_dir)
    print("Runtime run_dir:", runtime_run_dir)
    print("Exists:", runtime_run_dir.exists())

    if runtime_run_dir.exists():
        runtime_files = [p.relative_to(runtime_run_dir) for p in sorted(runtime_run_dir.rglob("*")) if p.is_file()]
        print("\nFiles:")
        for rel in runtime_files:
            print("-", rel)

        timeline_candidates = [
            runtime_run_dir / "timeline.jsonl",
            runtime_run_dir / "timeline.json",
            runtime_run_dir / "artifacts" / "runtime_timeline.json",
        ]
        existing_timeline_files = [p for p in timeline_candidates if p.exists()]

        print("\nTimeline candidates found:")
        if existing_timeline_files:
            for p in existing_timeline_files:
                print("-", p.relative_to(runtime_run_dir))
        else:
            print("None")

        timeline_jsonl = runtime_run_dir / "timeline.jsonl"
        if timeline_jsonl.exists():
            lines = [ln for ln in timeline_jsonl.read_text(encoding="utf-8").splitlines() if ln.strip()]
            print("\ntimeline.jsonl rows:", len(lines))
            if lines:
                print("Last row preview:")
                print(lines[-1][:800])

        if load_session_metrics is not None:
            try:
                runtime_metrics = load_session_metrics(str(runtime_run_dir))
                print("\nShared timeline replay succeeded.")
                pprint(runtime_metrics)
            except TypeError:
                try:
                    airway_order = getattr(runtime, "AIRWAY_VISIT_ORDER", None)
                    runtime_metrics = load_session_metrics(str(runtime_run_dir), airway_order)
                    print("\nShared timeline replay succeeded.")
                    pprint(runtime_metrics)
                except Exception as e:
                    print("\nShared timeline replay failed:", repr(e))
            except Exception as e:
                print("\nShared timeline replay failed:", repr(e))
        else:
            print("\nShared timeline loader is unavailable in this environment.")


## 3) Research pipeline

This pipeline uses the research manager to organize **multi-agent reasoning** and generate structured guidance.

### Key alignment points in the tightened research line
- the **multi-agent structure is preserved**
- shared curriculum drives `next_airway` and landmark truth
- shared event signal should produce `flag / ema / reason / soft_prompt`
- shared directional hint should appear in the returned `statepacket`
- statistics remains a **side-channel** for later performance analysis and reporting

### Research 流程
这个流程通过 research manager 组织多代理推理，并生成结构化指导内容。  
这里保留了真正的 multi-agent system 结构，同时把 shared backbone 接到了 research line 里。

In [ ]:
import json

def make_research_prompt(
    current_situation: str,
    previous_msgs: str = "",
    student_question: str = "",
) -> str:
    return f'''
CURRENT_SITUATION:
{current_situation.strip()}

PREVIOUS_MSGS:
{previous_msgs.strip()}

STUDENT_QUESTION:
{student_question.strip()}
'''.strip()


def print_research_result(result: dict):
    print("=" * 88)

    print("UI TEXT:")
    print(result.get("ui_text", ""))

    print("\nCURRICULUM PROGRESS:")
    pprint(result.get("curriculum_progress", {}))

    print("\nLANDMARK HINT:")
    pprint(result.get("landmark_hint", {}))

    print("\nEVENT PACKET:")
    pprint(result.get("event_packet", {}))

    statepacket = result.get("statepacket", {}) or {}
    print("\nSTATEPACKET (selected):")
    pprint({
        "anatomical_position": statepacket.get("anatomical_position"),
        "current_target": statepacket.get("current_target"),
        "next_destination": statepacket.get("next_destination"),
        "drift_detected": statepacket.get("drift_detected"),
        "need_recenter": statepacket.get("need_recenter"),
        "control_hint": statepacket.get("control_hint"),
    })

    print("\nANALYTICS (statistics side-channel):")
    analytics = result.get("analytics", {}) or {}
    pprint(analytics.get("statistics", result.get("statistics", {})))

### Part 1: Single-turn reasoning

In a single turn, the research manager takes the current input and produces structured guidance for that moment.

This case is intentionally shaped to exercise the shared backbone:
- curriculum progression from RB2 toward RB3
- slight drift / not centered
- raw control triplet via `m_jointsVelRel`
- event-signal-like cues via `distance_delta` and `heading_error`

In [ ]:
single_turn_situation = '''
Current region: RB2
Target region: RB3
Current airway: RB2
Target airway: RB3
The target is not yet centered and minor drift is present.
The view is slightly off-axis and the learner should re-center before advancing.
m_jointsVelRel=[0.0, 0.18, 0.0, 0.10, 0.0]
distance_delta=-0.30
heading_error=0.22
reached_regions=["RB1","RB2"]
'''.strip()

single_turn_prompt = make_research_prompt(
    current_situation=single_turn_situation,
    previous_msgs="",
    student_question="What should I do next?"
)

if manager is not None:
    try:
        single_turn_result = manager.run(single_turn_prompt)
        print_research_result(single_turn_result)
    except Exception as e:
        print("Research single-turn execution failed:")
        print(repr(e))
else:
    print("Skipping research single-turn run.")

### Part 2: Multi-turn structured generation

Across multiple turns, the pipeline generates structured outputs that can be stored, tracked, and used to support continuous multi-turn interaction.

What to watch:
- `curriculum_progress.next_airway` should move coherently
- `statepacket.next_destination` should stay aligned
- event packets should appear when the situation worsens
- statistics can accumulate as a side-channel without controlling the utterance

In [ ]:
demo_turns = [
    {
        "current_situation": '''
Current region: RB1
Target region: RB2
Current airway: RB1
Target airway: RB2
The learner has reached RB1 and is preparing to transition.
The view is stable but not yet aligned for the next branch.
m_jointsVelRel=[0.0, 0.06, 0.0, 0.00, 0.0]
distance_delta=-0.05
heading_error=0.08
reached_regions=["RB1"]
'''.strip(),
        "student_question": "How should I move toward the next airway?"
    },
    {
        "current_situation": '''
Current region: RB2
Target region: RB3
Current airway: RB2
Target airway: RB3
The learner has entered RB2, but the target is not centered and minor drift is present.
m_jointsVelRel=[0.0, 0.20, 0.0, 0.12, 0.0]
distance_delta=-0.28
heading_error=0.19
reached_regions=["RB1","RB2"]
'''.strip(),
        "student_question": "I can see the branch but I am drifting. What now?"
    },
    {
        "current_situation": '''
Current region: RB3
Target region: RB3
Current airway: RB3
Target airway: RB3
The learner is at the target airway. The view is centered and stable.
m_jointsVelRel=[0.0, 0.00, 0.0, 0.00, 0.0]
distance_delta=0.01
heading_error=0.02
reached_regions=["RB1","RB2","RB3"]
'''.strip(),
        "student_question": "Am I stable enough to continue?"
    },
]

research_results = []

if manager is not None:
    previous_msgs = ""
    for i, turn in enumerate(demo_turns, 1):
        print("\n" + "#" * 88)
        print(f"RESEARCH TURN {i}")

        prompt = make_research_prompt(
            current_situation=turn["current_situation"],
            previous_msgs=previous_msgs,
            student_question=turn["student_question"],
        )

        out = manager.run(prompt)
        research_results.append(out)

        print_research_result(out)

        ui_text = (out.get("ui_text") or "").strip()
        if ui_text:
            previous_msgs = (previous_msgs + "\n" + ui_text).strip()
else:
    print("Skipping multi-turn research demo.")

## 4) Logging and replay checks

The tightened logging rule is:
- default root = current working directory / `mas_log`
- the manager should expose its run directory

These cells check where the research line wrote its files and whether the shared timeline reader can load them back.

In [ ]:
if manager is not None and hasattr(manager, "logger"):
    run_dir = Path(manager.logger.run_dir)
    print("Research run_dir:", run_dir)
    print("Exists:", run_dir.exists())

    if run_dir.exists():
        print("\nFiles:")
        for p in sorted(run_dir.rglob("*")):
            if p.is_file():
                print("-", p.relative_to(run_dir))
else:
    print("No research logger available.")

In [ ]:
if manager is not None and load_session_metrics is not None and hasattr(manager, "logger"):
    run_dir = Path(manager.logger.run_dir)
    try:
        metrics = load_session_metrics(str(run_dir), manager.AIRWAY_VISIT_ORDER)
        print("Metrics loaded successfully.")
        pprint(metrics)
    except Exception as e:
        print("Timeline replay failed:", repr(e))
else:
    print("Skipping timeline replay check.")

## 5) Optional report check

This is useful for checking whether the reporting path stays aligned with the tightened research line.

In [ ]:
if manager is not None and hasattr(manager, "get_report") and hasattr(manager, "logger"):
    run_dir = Path(manager.logger.run_dir)
    try:
        report_text = manager.get_report(str(run_dir))
        print(report_text[:4000])
    except Exception as e:
        print("Report generation failed:", repr(e))
else:
    print("Skipping report generation.")

## Notes

When you use this notebook as a demo, the key interpretation is:

- **runtime output** = fast real-time guidance text
- **research output** = multi-agent structured guidance
- **curriculum / landmark / event / directional hint** = grounded shared backbone
- **statistics** = side-channel analytics for later assessment
- **multi-agent structure is preserved** even if not every agent fully shapes the real-time utterance yet